# 01 — Data Exploration

Load `CUADv1.json`, compute per-category support counts across the 510 contracts, and confirm the 10-category scope shortlist (see README) against real class-imbalance numbers.

In [ ]:
import json
import re
from collections import defaultdict

with open("../data/CUADv1.json", encoding="utf-8") as f:
    data = json.load(f)

contract_categories_present = defaultdict(set)
all_contracts = set()
category_pattern = re.compile(r'related to "([^"]+)"')

for entry in data["data"]:
    title = entry["title"]
    all_contracts.add(title)
    for para in entry["paragraphs"]:
        for qa in para["qas"]:
            m = category_pattern.search(qa["question"])
            if not m:
                continue
            category = m.group(1)
            is_impossible = qa.get("is_impossible", False)
            has_answer = len(qa.get("answers", [])) > 0
            if has_answer and not is_impossible:
                contract_categories_present[category].add(title)

total_contracts = len(all_contracts)
print(f"Total contracts: {total_contracts}")
print(f"Total categories found: {len(contract_categories_present)}")

In [ ]:
counts = sorted(
    ((cat, len(titles)) for cat, titles in contract_categories_present.items()),
    key=lambda x: -x[1],
)
for cat, cnt in counts:
    print(f"{cnt:4d}  ({100 * cnt / total_contracts:5.1f}%)  {cat}")

## Selected 10 categories

Governing Law, Anti-Assignment, Cap On Liability, License Grant, Audit Rights, Termination For Convenience, Exclusivity, Change Of Control, Non-Compete, Uncapped Liability — see README for rationale. Confirm these counts still hold and bring any surprises to the advisor meeting.